# Model Selection - Small Open-Source LLMs

**Colab T4 GPU** - pure transformers, no extra libraries

Models tested: Qwen2.5-3B, Phi-3.5-mini (3.8B), Qwen3-4B, Qwen2.5-7B

Strategy: few-shot example in every prompt + up to 3 retries (greedy, temp 0.3, temp 0.7)

Correct answer for test case: **B - Pulmonary embolism**

## 1. Install & Setup

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
!pip install -q transformers accelerate bitsandbytes
import torch, gc, json, re, time, textwrap, traceback
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
print(f"torch: {torch.__version__}")
print(f"CUDA:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:   {torch.cuda.get_device_name(0)}")
    print(f"VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Models & Prompts

In [ ]:
MODELS = [
    "Qwen/Qwen2.5-3B-Instruct",        # 3B   - Qwen
    "microsoft/Phi-3.5-mini-instruct",  # 3.8B - Microsoft
    "Qwen/Qwen3-4B",                    # 4B   - newest Qwen (thinking off)
]

CORRECT = 'B'  # Pulmonary embolism

EHR      = '45-year-old male presenting with: sudden-onset pleuritic chest pain and dyspnoea'
QUESTION = 'What is the most likely diagnosis?'
OPTIONS  = 'A. Myocardial infarction  B. Pulmonary embolism  C. GERD  D. Anxiety disorder'

EXAMPLE_PATIENT  = '60-year-old female with crushing substernal chest pain radiating to left arm.'
EXAMPLE_QUESTION = 'What is the most likely diagnosis?'
EXAMPLE_OPTIONS  = 'A. GERD  B. Panic attack  C. Myocardial infarction  D. Costochondritis'

SCHEMA1_PROMPT = (
    'You are an experienced clinician. Respond with ONLY a JSON object - '
    'no explanation, no markdown, no extra text.\n\n'
    'The JSON must have EXACTLY these three keys:\n'
    '  "clarifying_question" - one short clinical question (string)\n'
    '  "preliminary_assessment" - one letter: A, B, C or D (string)\n'
    '  "confidence" - integer between 0 and 100\n\n'
    'EXAMPLE (different patient):\n'
    f'Patient: {EXAMPLE_PATIENT}\n'
    f'Question: {EXAMPLE_QUESTION}\n'
    f'Options: {EXAMPLE_OPTIONS}\n'
    '{"clarifying_question": "Any history of coronary artery disease?", '
    '"preliminary_assessment": "C", "confidence": 85}\n\n'
    'NOW YOUR PATIENT:\n'
    f'Patient: {EHR}\n'
    f'Question: {QUESTION}\n'
    f'Options: {OPTIONS}\n'
)

SCHEMA2_PROMPT = (
    'You are an experienced clinician. You asked one clarifying question and received '
    'the patient answer. Respond with ONLY a JSON object - no explanation, no markdown.\n\n'
    'The JSON must have EXACTLY these three keys:\n'
    '  "updated_assessment" - one letter: A, B, C or D (string)\n'
    '  "confidence" - integer between 0 and 100\n'
    '  "clarifying_question" - your NEXT clinical question (string)\n\n'
    'EXAMPLE (different patient):\n'
    f'Patient: {EXAMPLE_PATIENT}\n'
    'Patient answer: The pain started 30 min ago and radiates to my jaw.\n'
    '{"updated_assessment": "C", "confidence": 90, '
    '"clarifying_question": "Any nausea or sweating?"}\n\n'
    'NOW YOUR PATIENT:\n'
    f'Patient: {EHR}\n'
    f'Question: {QUESTION}\n'
    f'Options: {OPTIONS}\n'
    'Patient answer: Pain started 2 hours ago, worse when breathing in.\n'
)

SCHEMA3_PROMPT = (
    'You are an experienced clinician. You have all the information you need. '
    'Respond with ONLY a JSON object - no explanation, no markdown, no extra text.\n\n'
    'The JSON must have EXACTLY these two keys:\n'
    '  "updated_assessment" - one letter: A, B, C or D (string)\n'
    '  "updated_confidence" - integer between 0 and 100\n\n'
    'EXAMPLE (different patient):\n'
    f'Patient: {EXAMPLE_PATIENT}\n'
    'Final info: ECG shows ST elevation. Troponin elevated.\n'
    '{"updated_assessment": "C", "updated_confidence": 97}\n\n'
    'NOW YOUR PATIENT:\n'
    f'Patient: {EHR}\n'
    f'Question: {QUESTION}\n'
    f'Options: {OPTIONS}\n'
    'Patient final answer: No radiation to arm/jaw. HR 110, SpO2 92%.\n'
)

SCHEMA1_KEYS = {'clarifying_question', 'preliminary_assessment', 'confidence'}
SCHEMA2_KEYS = {'updated_assessment', 'confidence', 'clarifying_question'}
SCHEMA3_KEYS = {'updated_assessment', 'updated_confidence'}

print(f"Models  : {len(MODELS)}")
print(f"Correct : {CORRECT} - Pulmonary embolism")

## 3. Helper Functions

In [ ]:
def extract_json(text):
    # Strip DeepSeek think blocks
    text = re.sub('<think>[\\s\\S]*?</think>', '', text, flags=re.IGNORECASE).strip()
    # Strip markdown fences
    text = re.sub('```(?:json)?', '', text).replace('```', '').strip()
    start = text.find('{')
    if start == -1:
        return None
    depth, i = 0, start
    while i < len(text):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i+1])
                except json.JSONDecodeError:
                    return None
        i += 1
    return None


def generate(model, tokenizer, prompt, max_new_tokens=300, temperature=None):
    # Step 1: format prompt string via chat template (tokenize=False -> always returns str)
    messages = [{'role': 'user', 'content': prompt}]
    try:
        try:
            # Qwen3: disable built-in thinking mode
            text = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True,
                tokenize=False, enable_thinking=False
            )
        except TypeError:
            text = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False
            )
    except Exception:
        text = 'User: ' + prompt + '\nAssistant:'
    # Step 2: tokenize the string -> guaranteed plain tensor
    input_ids = tokenizer(text, return_tensors='pt').input_ids.to(model.device)

    pad_id = tokenizer.eos_token_id or tokenizer.pad_token_id or 0
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=pad_id,
        return_dict_in_generate=False,
    )
    if temperature is None:
        gen_kwargs['do_sample'] = False
    else:
        gen_kwargs['do_sample'] = True
        gen_kwargs['temperature'] = temperature
        gen_kwargs['top_p'] = 0.9

    with torch.no_grad():
        out = model.generate(input_ids, **gen_kwargs)

    # out may be a tensor or a special generate output object
    if not isinstance(out, torch.Tensor):
        out = out.sequences
    new_tokens = out[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def run_with_retry(model, tokenizer, prompt, required_keys, max_tries=3):
    temps = [None, 0.3, 0.7]
    raw = ''
    for attempt, temp in enumerate(temps[:max_tries], 1):
        try:
            raw = generate(model, tokenizer, prompt, temperature=temp)
        except Exception:
            raw = '[generate error] ' + traceback.format_exc()[-300:]
            continue
        parsed = extract_json(raw)
        if parsed and required_keys.issubset(parsed.keys()):
            return parsed, raw, attempt
    return None, raw, max_tries


def free_gpu(*objects):
    for obj in objects:
        if obj is not None:
            del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        used  = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  GPU freed  VRAM: {used:.2f} / {total:.1f} GB")


def test_model(model_id):
    print(f"\n{'='*65}\nTesting: {model_id}\n{'='*65}")
    result = dict(
        model=model_id, loaded=False, error='',
        s1=None, s1_raw='', s1_tries=0, s1_time=None,
        s2=None, s2_raw='', s2_tries=0, s2_time=None,
        s3=None, s3_raw='', s3_tries=0, s3_time=None,
    )
    model = tokenizer = None
    try:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb,
            device_map='auto', trust_remote_code=True,
        )
        model.eval()
        result['loaded'] = True
        if torch.cuda.is_available():
            print(f"VRAM after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

        # --- smoke test ---
        print('Running smoke test...')
        _ = generate(model, tokenizer, 'Reply with: ok', max_new_tokens=5)
        print('Smoke test passed.')

        for schema_id, prompt, keys, letter_key, conf_key in [
            ('Schema 1 - Turn 0',    SCHEMA1_PROMPT, SCHEMA1_KEYS, 'preliminary_assessment', 'confidence'),
            ('Schema 2 - Continuation', SCHEMA2_PROMPT, SCHEMA2_KEYS, 'updated_assessment',   'confidence'),
            ('Schema 3 - Final',     SCHEMA3_PROMPT, SCHEMA3_KEYS, 'updated_assessment',   'updated_confidence'),
        ]:
            sk = 's' + schema_id[7]  # 's1', 's2', 's3'
            print(f'\n--- {schema_id} ---')
            t0 = time.time()
            parsed, raw, tries = run_with_retry(model, tokenizer, prompt, keys)
            result[sk+'_time'] = round(time.time() - t0, 1)
            result[sk+'_raw'] = raw
            result[sk+'_tries'] = tries
            if parsed:
                letter = str(parsed.get(letter_key, '')).strip().upper()
                conf   = parsed.get(conf_key, -1)
                if letter in {'A','B','C','D'} and isinstance(conf,(int,float)) and 0<=conf<=100:
                    result[sk] = parsed
                    flag = 'CORRECT' if letter == CORRECT else f'WRONG (expected {CORRECT})'
                    print(f'  PASS  {letter} ({flag})  conf={conf}  tries={tries}  {result[sk+'_time']}s')
                    if 'clarifying_question' in parsed:
                        print(f'  CQ: {str(parsed["clarifying_question"])[:120]}')
                else:
                    print(f'  FAIL bad values: letter={letter!r} conf={conf!r}')
            else:
                print(f'  FAIL no valid JSON after {tries} tries')
                print(f'  Raw[:300]: {raw[:300]}')

    except Exception:
        tb = traceback.format_exc()
        result['error'] = tb
        print(f'ERROR:\n{tb}')
    finally:
        free_gpu(model, tokenizer)

    passed = sum(1 for k in ('s1','s2','s3') if result[k])
    total_t = sum(t for t in (result['s1_time'],result['s2_time'],result['s3_time']) if t)
    print(f'\nResult: {passed}/3 schemas passed  |  total: {total_t:.1f}s')
    return result

print('Helpers defined.')

## 4. Run Tests

*~3-10 min per model*

In [ ]:
all_results = []
for model_id in MODELS:
    res = test_model(model_id)
    all_results.append(res)
    print()
print("All models tested.")

## 5. Results Summary

In [ ]:
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"  {'Model':<38} {'S1':>5} {'S2':>5} {'S3':>5}  Pass  Time    Status")
print("-"*80)

rows = []
for r in all_results:
    name = r["model"].split("/")[-1]
    if not r["loaded"]:
        print(f"  {name:<38}   ERR   ERR   ERR   -/-   LOAD ERROR")
        continue

    def cs(sk, lk):
        d = r[sk]
        if d is None: return '  X'
        let = str(d.get(lk,'?')).strip().upper()
        return let + ('v' if let == CORRECT else '~')

    s1s = cs('s1','preliminary_assessment')
    s2s = cs('s2','updated_assessment')
    s3s = cs('s3','updated_assessment')
    passed = sum(1 for k in ('s1','s2','s3') if r[k])
    total_t = sum(t for t in (r['s1_time'],r['s2_time'],r['s3_time']) if t)
    n_correct = sum([
        bool(r['s1']) and str(r['s1'].get('preliminary_assessment','')).upper()==CORRECT,
        bool(r['s2']) and str(r['s2'].get('updated_assessment','')).upper()==CORRECT,
        bool(r['s3']) and str(r['s3'].get('updated_assessment','')).upper()==CORRECT,
    ])
    status = 'ALL PASS' if passed==3 else ('PARTIAL' if passed>0 else 'ALL FAIL')
    print(f'  {name:<38} {s1s:>5} {s2s:>5} {s3s:>5}  {passed}/3  {total_t:>5.1f}s  {status}')
    rows.append((passed, n_correct, -total_t, r['model']))

print("="*80)
rows.sort(reverse=True)
print("\nRanking (schemas passed -> correct diagnoses -> fastest):")
for i,(p,c,t,m) in enumerate(rows, 1):
    print(f'  {i}. {m.split("/")[-1]:<40}  {p}/3 schemas  {c}/3 correct  {-t:.1f}s')
print("\nLegend: Bv=correct  A~/C~=wrong diagnosis  X=schema fail")

## 6. Raw Output Inspection (failing schemas)

In [ ]:
for r in all_results:
    if not r['loaded']: continue
    any_fail = any(r[k] is None for k in ('s1','s2','s3'))
    if not any_fail: continue
    print(f"\n{'='*60}\nModel: {r['model']}")
    if r['error']:
        print(f"  Traceback:\n{r['error'][:600]}")
        continue
    for sk, label in [('s1','Schema 1'),('s2','Schema 2'),('s3','Schema 3')]:
        if r[sk] is None:
            raw = r[sk+'_raw'][:500]
            print(f"\n  [{label} FAIL  {r[sk+'_tries']} tries] Raw output:")
            print(textwrap.indent(raw, '    '))

## 7. Save Results

In [ ]:
SAVE_PATH = '/content/model_selection_results.json'
with open(SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
print(f'Saved to: {SAVE_PATH}')
print('Download: Files panel (left sidebar) -> right-click -> Download')